# Lab 6 — Neural Networks & Backpropagation: SOLUTION

This notebook works through the **Neural Networks Tutorial** exercises from the PDF.

## Network Architecture

```
Inputs       Hidden Layer      Output Layer

  x₁ ──w₁₁──▶ h₁ ──v₁₁──▶ y₁
    ╲──w₂₁──▶    ╲──v₂₁──▶
    ╱──w₁₂──▶    ╱──v₁₂──▶
  x₂ ──w₂₂──▶ h₂ ──v₂₂──▶ y₂
```

- **Activation function:** Sigmoid σ(z) = 1 / (1 + e^−z)
- **Loss function:** L = (y₁ − t₁)² + (y₂ − t₂)²

## Given Weights

| | x₁ | x₂ |
|--|------|------|
| **→ h₁** | w₁₁ = 6 | w₁₂ = −2 |
| **→ h₂** | w₂₁ = −3 | w₂₂ = 5 |

| | h₁ | h₂ |
|--|------|------|
| **→ y₁** | v₁₁ = 1 | v₁₂ = 0.25 |
| **→ y₂** | v₂₁ = −2 | v₂₂ = 2 |

## Training Data

| Example | x₁ | x₂ | Target t₁ | Target t₂ |
|---------|----|----|-----------|----------|
| 1 | 3 | 1 | 1 | 0 |
| 2 | −1 | 4 | 0 | 1 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Setup ---
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_deriv(z):
    """Derivative of sigmoid: σ(z) * (1 - σ(z))"""
    s = sigmoid(z)
    return s * (1 - s)

# Weight matrices (row i = weights INTO unit i)
W = np.array([[6.0, -2.0],   # weights into h1: [w11, w12]
              [-3.0, 5.0]])   # weights into h2: [w21, w22]

V = np.array([[1.0,  0.25],  # weights into y1: [v11, v12]
              [-2.0, 2.0]])   # weights into y2: [v21, v22]

# Training data
X = np.array([[3, 1],   # example 1
              [-1, 4]]) # example 2
T = np.array([[1, 0],   # targets for example 1
              [0, 1]])  # targets for example 2

print("Setup complete.")
print(f"W (input → hidden):\n{W}")
print(f"\nV (hidden → output):\n{V}")

## Part 1: Forward Pass

For each example, compute hidden activations then output activations.

**Step 1:** Pre-activation of hidden layer: `z_h = W @ x`

**Step 2:** Hidden activation: `h = σ(z_h)`

**Step 3:** Pre-activation of output layer: `z_y = V @ h`

**Step 4:** Output activation: `y = σ(z_y)`

In [ ]:
def forward_pass(x, W, V, verbose=True):
    """Compute full forward pass and return all intermediate values."""
    # Hidden layer
    z_h = W @ x          # pre-activation
    h   = sigmoid(z_h)   # activation

    # Output layer
    z_y = V @ h          # pre-activation
    y   = sigmoid(z_y)   # output

    if verbose:
        print(f"  Input x = {x}")
        print(f"  z_h = W @ x = {z_h}")
        print(f"  h   = σ(z_h) = {np.round(h, 4)}")
        print(f"  z_y = V @ h = {np.round(z_y, 4)}")
        print(f"  y   = σ(z_y) = {np.round(y, 4)}")

    return z_h, h, z_y, y

print("=== Example 1: x=[3,1], target=[1,0] ===")
z_h1, h1, z_y1, y1 = forward_pass(X[0], W, V)

print("\n=== Example 2: x=[-1,4], target=[0,1] ===")
z_h2, h2, z_y2, y2 = forward_pass(X[1], W, V)

## Part 2: Loss Calculation

**Loss function:** L = (y₁ − t₁)² + (y₂ − t₂)²

This is the sum of squared errors between the network's predictions and the targets.

In [ ]:
def compute_loss(y, t):
    return np.sum((y - t) ** 2)

L1 = compute_loss(y1, T[0])
L2 = compute_loss(y2, T[1])

print("=== Loss Calculation ===")
print(f"\nExample 1:")
print(f"  y = {np.round(y1, 4)},  t = {T[0]}")
print(f"  L = ({y1[0]:.4f} - {T[0][0]})² + ({y1[1]:.4f} - {T[0][1]})²")
print(f"  L = {(y1[0]-T[0][0])**2:.4f} + {(y1[1]-T[0][1])**2:.4f} = {L1:.4f}")

print(f"\nExample 2:")
print(f"  y = {np.round(y2, 4)},  t = {T[1]}")
print(f"  L = ({y2[0]:.4f} - {T[1][0]})² + ({y2[1]:.4f} - {T[1][1]})²")
print(f"  L = {(y2[0]-T[1][0])**2:.4f} + {(y2[1]-T[1][1])**2:.4f} = {L2:.4f}")

print(f"\nTotal loss (both examples): {L1 + L2:.4f}")

## Part 3: Partial Derivatives Exercise

Before backpropagation, practice computing partial derivatives.

**Function:** f(x, y) = 3x² + 2xy + 5

**Partial derivatives:**
$$\frac{\partial f}{\partial x} = 6x + 2y$$
$$\frac{\partial f}{\partial y} = 2x$$

**Gradient at a point:**
$$\nabla f = \left(\frac{\partial f}{\partial x},\ \frac{\partial f}{\partial y}\right)$$

In [ ]:
# f(x,y) = 3x^2 + 2xy + 5
def f(x, y):
    return 3*x**2 + 2*x*y + 5

def df_dx(x, y):
    return 6*x + 2*y  # partial derivative with respect to x

def df_dy(x, y):
    return 2*x         # partial derivative with respect to y

# Evaluate at several points
test_points = [(1, 2), (0, 3), (-1, 1), (2, -1)]

print("Partial derivatives of f(x,y) = 3x² + 2xy + 5")
print(f"  ∂f/∂x = 6x + 2y")
print(f"  ∂f/∂y = 2x")
print()
print(f"{'Point':>12} | {'f(x,y)':>8} | {'∂f/∂x':>8} | {'∂f/∂y':>8} | {'Gradient':>16}")
print("-" * 65)
for (x, y) in test_points:
    grad = (df_dx(x, y), df_dy(x, y))
    print(f"  ({x:>2}, {y:>2})     | {f(x,y):>8.1f} | {df_dx(x,y):>8.1f} | {df_dy(x,y):>8.1f} | ({grad[0]:>6.1f}, {grad[1]:>6.1f})")

## Part 4: Backpropagation

Backpropagation uses the **chain rule** to compute how the loss changes with respect to each weight.

### Chain Rule for Output Weights (V)

For output weight $v_{ij}$ (connecting hidden unit $j$ to output unit $i$):

$$\frac{\partial L}{\partial v_{ij}} = \frac{\partial L}{\partial y_i} \cdot \frac{\partial y_i}{\partial z_{y_i}} \cdot \frac{\partial z_{y_i}}{\partial v_{ij}}$$

Where:
- $\frac{\partial L}{\partial y_i} = 2(y_i - t_i)$ (from loss function)
- $\frac{\partial y_i}{\partial z_{y_i}} = \sigma'(z_{y_i}) = y_i(1 - y_i)$ (sigmoid derivative)
- $\frac{\partial z_{y_i}}{\partial v_{ij}} = h_j$ (the input to this weight)

### Chain Rule for Hidden Weights (W)

For hidden weight $w_{ij}$:

$$\frac{\partial L}{\partial w_{ij}} = \left(\sum_k \frac{\partial L}{\partial z_{y_k}} \cdot v_{ki}\right) \cdot \sigma'(z_{h_i}) \cdot x_j$$

In [ ]:
def backprop(x, t, W, V, verbose=True):
    """Compute gradients for W and V using backpropagation."""
    # --- Forward pass (save all intermediates) ---
    z_h = W @ x
    h   = sigmoid(z_h)
    z_y = V @ h
    y   = sigmoid(z_y)

    # --- Output layer gradients ---
    dL_dy  = 2 * (y - t)                     # ∂L/∂y  shape (2,)
    dy_dzy = sigmoid_deriv(z_y)              # σ'(z_y)  shape (2,)
    delta_y = dL_dy * dy_dzy                 # ∂L/∂z_y  shape (2,)

    # Gradient for V: outer product of delta_y and h
    dL_dV = np.outer(delta_y, h)             # shape (2, 2)

    # --- Hidden layer gradients ---
    dzy_dh  = V.T                            # V transposed  shape (2, 2)
    dL_dh   = dzy_dh @ delta_y              # ∂L/∂h  shape (2,)
    dh_dzh  = sigmoid_deriv(z_h)            # σ'(z_h)  shape (2,)
    delta_h = dL_dh * dh_dzh               # ∂L/∂z_h  shape (2,)

    # Gradient for W: outer product of delta_h and x
    dL_dW = np.outer(delta_h, x)             # shape (2, 2)

    if verbose:
        print(f"  x = {x},  t = {t}")
        print(f"  y = {np.round(y, 4)}")
        print(f"  dL/dy  = 2*(y-t) = {np.round(dL_dy, 4)}")
        print(f"  σ'(z_y) = y*(1-y) = {np.round(dy_dzy, 4)}")
        print(f"  delta_y = dL/dy * σ'(z_y) = {np.round(delta_y, 4)}")
        print(f"\n  dL/dV (gradient for output weights):\n{np.round(dL_dV, 4)}")
        print(f"\n  dL/dh = V.T @ delta_y = {np.round(dL_dh, 4)}")
        print(f"  σ'(z_h) = h*(1-h) = {np.round(dh_dzh, 4)}")
        print(f"  delta_h = {np.round(delta_h, 4)}")
        print(f"\n  dL/dW (gradient for hidden weights):\n{np.round(dL_dW, 4)}")

    return dL_dW, dL_dV

print("=== Backpropagation — Example 1 ===")
dW1, dV1 = backprop(X[0], T[0], W, V)

print("\n=== Backpropagation — Example 2 ===")
dW2, dV2 = backprop(X[1], T[1], W, V)

## Part 5: Weight Update (Gradient Descent)

Once we have the gradients, we update the weights:

$$W_{\text{new}} = W_{\text{old}} - \alpha \cdot \frac{\partial L}{\partial W}$$

Where **α** (alpha) is the **learning rate** — how big a step to take.

We accumulate gradients over both training examples before updating.

In [ ]:
alpha = 0.1  # learning rate

# Accumulate gradients over both examples
total_dW = dW1 + dW2
total_dV = dV1 + dV2

print(f"Total gradient for W (sum over both examples):\n{np.round(total_dW, 4)}")
print(f"\nTotal gradient for V (sum over both examples):\n{np.round(total_dV, 4)}")

# Update weights
W_new = W - alpha * total_dW
V_new = V - alpha * total_dV

print(f"\n--- After one gradient descent step (α={alpha}) ---")
print(f"W_old:\n{W}")
print(f"W_new:\n{np.round(W_new, 4)}")
print(f"\nV_old:\n{V}")
print(f"V_new:\n{np.round(V_new, 4)}")

# Check: did the loss decrease?
_, _, _, y1_new = forward_pass(X[0], W_new, V_new, verbose=False)
_, _, _, y2_new = forward_pass(X[1], W_new, V_new, verbose=False)
L_before = compute_loss(y1, T[0]) + compute_loss(y2, T[1])
L_after  = compute_loss(y1_new, T[0]) + compute_loss(y2_new, T[1])
print(f"\nTotal loss before update: {L_before:.4f}")
print(f"Total loss after  update: {L_after:.4f}")
print(f"Loss {'decreased ✅' if L_after < L_before else 'increased ❌'}")

## Part 6: Full Training Loop (200 Epochs)

Now let's train the network for 200 epochs and watch the loss decrease.

In [ ]:
# Reset weights to original values
W_train = np.array([[6.0, -2.0], [-3.0, 5.0]])
V_train = np.array([[1.0, 0.25], [-2.0, 2.0]])

alpha = 0.1
epochs = 200
loss_history = []

for epoch in range(epochs):
    total_dW = np.zeros_like(W_train)
    total_dV = np.zeros_like(V_train)
    total_loss = 0

    # Accumulate gradients over all training examples
    for i in range(len(X)):
        x, t = X[i], T[i]

        # Forward pass
        z_h = W_train @ x
        h   = sigmoid(z_h)
        z_y = V_train @ h
        y   = sigmoid(z_y)

        total_loss += compute_loss(y, t)

        # Backprop
        delta_y = 2 * (y - t) * sigmoid_deriv(z_y)
        dL_dV   = np.outer(delta_y, h)

        delta_h = (V_train.T @ delta_y) * sigmoid_deriv(z_h)
        dL_dW   = np.outer(delta_h, x)

        total_dW += dL_dW
        total_dV += dL_dV

    # Update weights
    W_train -= alpha * total_dW
    V_train -= alpha * total_dV

    loss_history.append(total_loss)

    if epoch % 20 == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch+1:>3}: Loss = {total_loss:.4f}")

print(f"\nFinal W:\n{np.round(W_train, 4)}")
print(f"\nFinal V:\n{np.round(V_train, 4)}")

In [ ]:
# Verify final predictions
print("=== Final Predictions After Training ===")
for i in range(len(X)):
    _, _, _, y_pred = forward_pass(X[i], W_train, V_train, verbose=False)
    print(f"\nExample {i+1}: x={X[i]}, target={T[i]}")
    print(f"  Predicted: {np.round(y_pred, 4)}")
    print(f"  Rounded:   {np.round(y_pred).astype(int)}")
    print(f"  Correct:   {'✅' if np.all(np.round(y_pred) == T[i]) else '❌'}")

In [ ]:
# Plot training loss curve
plt.figure(figsize=(9, 4))
plt.plot(range(1, epochs + 1), loss_history, 'b-', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Total Loss')
plt.title('Training Loss over 200 Epochs')
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.axhline(y=loss_history[-1], color='r', linestyle='--', alpha=0.7,
            label=f'Final loss = {loss_history[-1]:.4f}')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Initial loss: {loss_history[0]:.4f}")
print(f"Final loss:   {loss_history[-1]:.4f}")
print(f"Loss reduced by {(1 - loss_history[-1]/loss_history[0])*100:.1f}%")

## Summary: The Backpropagation Algorithm

1. **Forward pass** — compute activations layer by layer from input to output
2. **Compute loss** — measure how wrong the predictions are
3. **Backward pass** — use the chain rule to propagate error gradients from output back to input
4. **Update weights** — subtract a small step in the gradient direction
5. **Repeat** until loss is acceptably small

### Key Formulas

| Step | Formula |
|------|---------|
| Forward (hidden) | `h = σ(W @ x)` |
| Forward (output) | `y = σ(V @ h)` |
| Loss | `L = Σ(y - t)²` |
| Output delta | `δ_y = 2(y-t) ⊙ σ'(z_y)` |
| Gradient V | `∂L/∂V = outer(δ_y, h)` |
| Hidden delta | `δ_h = (V.T @ δ_y) ⊙ σ'(z_h)` |
| Gradient W | `∂L/∂W = outer(δ_h, x)` |
| Weight update | `W ← W - α · ∂L/∂W` |

---
*These are the exact same operations scaled up in every modern deep learning framework (PyTorch, TensorFlow).*